In [198]:
import os
from utils import extract_body, tokenize, clean_tokens, decode, is_manual_label_tag, is_auto_label_tag
from utils import chunk_tokens, flatten_token_chunks
from utils import extract_few_shot_examples
from utils import select_few_shot 
from utils import merge_tokens_with_auto_labels, add_style_and_parent_to_auto_labels, compare_html_allow_auto_labels
from utils import HTMLLabel
from utils.few_shot_utils import prepare_label_tokens
from models import GPTAssistant
from process_chunks import process_chunks

In [199]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
model_name = "gpt-4.1"

n_few_shot = 10  # Number of few-shot examples to use

#### Define the text to process, and where to save it

In [224]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "1997CanLII16226_ONCA"
anno = "llm"
version = "v1"
out_version = "v2"
html_path = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\{filename}_llm_{version}.html"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}"


# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


fs_filename = "1999CanLII7320_annotated"
fs_anno = "EG"
fs_version = "v1"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}_{fs_anno}_{fs_version}.html"
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA\1997CanLII16226_ONCA_llm_v1.html
   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\1999CanLII7320_annotated_EG_v1.html


### Process The HTML Content

In [238]:
# ---------- Tokenize html content ----------
tokens = tokenize(html_content)

In [250]:
# ---------- Tokenize html content ----------
fs_tokens = tokenize(fs_html_content)

In [251]:
sublabel_config = {
    "parent":["decision", "legislation", "secondary sources"], # only extract sublabels under these parents
    "already_labeled":[], # do not extract sublabels under these labels
    "new_labels":["title"],
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
}

# Do not use remove_labels here, as we need the parent labels to identify sublabels : This could  create issues.

#### Get few shot

In [228]:
def get_list_of_labels(tokens, label_type="auto_label"):
    result = []
    for token in tokens:
        if label_type == "auto_label" and is_auto_label_tag(token) == 1 : # This is a auto_label opening tag
            result.append(HTMLLabel(token))
        if label_type == "manual_label" and is_manual_label_tag(token) == 1 :
            result.append(HTMLLabel(token))
    return result

In [229]:
def get_list_of_mention(tokens, keep_labels, label_type=None):
    """
    Extract mentions from tokens and return their positions.
    
    Args:
        tokens: List of tokens to search
        keep_labels: List of label names to keep (e.g., ["title", "decision"])
        label_type: Optional filter - "manual_label", "auto_label", or None (both)
    
    Returns:
        List of tuples: (HTMLLabel object, start_index, end_index)
        - HTMLLabel object: The parsed opening tag
        - start_index: Index of the opening tag in tokens list
        - end_index: Index of the closing tag in tokens list
    """
    mentions = []
    i = 0
    
    while i < len(tokens):
        token = tokens[i]
        
        # Check if this matches the label type we're looking for
        is_match = False
        if label_type == "manual_label" and is_manual_label_tag(token) == 1:
            is_match = True
        elif label_type == "auto_label" and is_auto_label_tag(token) == 1:
            is_match = True
        elif label_type is None and (is_manual_label_tag(token) == 1 or is_auto_label_tag(token) == 1):
            is_match = True
        
        if is_match:
            html_label = HTMLLabel(token)
            
            # Check if this label is in keep_labels
            if html_label.name in keep_labels:
                start_index = i
                depth = 1
                i += 1
                
                # Find the matching closing tag
                while i < len(tokens) and depth > 0:
                    current_token = tokens[i]
                    
                    # Check if it's an opening tag of the same type
                    if label_type == "manual_label" and is_manual_label_tag(current_token) == 1:
                        depth += 1
                    elif label_type == "auto_label" and is_auto_label_tag(current_token) == 1:
                        depth += 1
                    elif label_type is None:
                        if is_manual_label_tag(current_token) == 1 or is_auto_label_tag(current_token) == 1:
                            depth += 1
                    
                    # Check if it's a closing tag of the same type
                    if label_type == "manual_label" and is_manual_label_tag(current_token) == 2:
                        depth -= 1
                    elif label_type == "auto_label" and is_auto_label_tag(current_token) == 2:
                        depth -= 1
                    elif label_type is None:
                        if is_manual_label_tag(current_token) == 2 or is_auto_label_tag(current_token) == 2:
                            depth -= 1
                    
                    if depth == 0:
                        end_index = i
                        mentions.append((html_label, start_index, end_index))
                        break
                    
                    i += 1
                continue
        
        i += 1
    
    return mentions

In [96]:
auto_tokens = get_list_of_labels(tokens = normalized_cleaned_tokens, label_type="auto_label")
manual_tokens = get_list_of_labels(tokens = normalized_cleaned_tokens, label_type="manual_label")
print(f"   ✓ Number of auto labels in document: {len(auto_tokens)}")
print(f"   ✓ Number of manual labels in document: {len(manual_tokens)}")

   ✓ Number of auto labels in document: 230
   ✓ Number of manual labels in document: 607


In [252]:
def extract_few_shot_examples_from_labels(tokens, sublabel_config):
    """
    Extract few-shot examples from parent labels containing sublabels.
    
    Args:
        tokens: List of tokens to process
        sublabel_config: Configuration dict with:
            - parent: List of parent label names to extract from
            - keep_labels: List of sublabel names to keep in output
            - keep_attributes, switch_type, use_simplified: Transform options
    
    Returns:
        List of tuples (input, output) where:
        - input: Parent label with only parent tag preserved
        - output: Parent label with both parent and specified sublabels preserved
    """
    examples = []
    
    # Get all mentions of parent labels using the utility function
    parent_mentions = get_list_of_mention(tokens=tokens, keep_labels=sublabel_config["parent"], label_type="manual_label")
    
    for _, start_idx, end_idx in parent_mentions:
        # Extract the mention tokens (from start to end inclusive)
        mention = tokens[start_idx:end_idx + 1]
        
        # Input: keep only parent labels
        input_legal_config = sublabel_config.copy()
        input_legal_config["keep_labels"] = sublabel_config["parent"] + sublabel_config["already_labeled"]
        input_tokens = prepare_label_tokens(mention, label_config=input_legal_config)
        
        # Output: keep both parent and sublabels
        output_legal_config = sublabel_config.copy()
        output_legal_config["keep_labels"] = sublabel_config["new_labels"] + sublabel_config["already_labeled"] + sublabel_config["parent"]
        output_tokens = prepare_label_tokens(mention, label_config=output_legal_config)
        
        examples.append((decode(input_tokens), decode(output_tokens)))
    
    return examples

In [243]:
import random
from utils import is_auto_label_tag
from utils.htmlLabel import from_simplified

def select_few_shot(examples, n, method="order", list_of_labels=None, distribution=None):
    """
    Select n few-shot examples from the provided list.
    
    Args:
        examples: List of tuples (input, expected_output)
        n: Number of examples to select
        method: Selection method - "order", "random", or "distributed"
        list_of_labels: List of label names for distributed selection (e.g., ["source"])
        distribution: List of proportions for each label (e.g., [0.5])
                     If sum < 1.0, remainder is filled with random "other" examples
    
    Returns:
        list: Selected few-shot examples
        
    Examples:
        # Select first 10 in order
        select_few_shot(examples, 10, method="order")
        
        # Select 10 with 50% containing "source" label, 50% random others
        select_few_shot(examples, 10, method="distributed", 
                       list_of_labels=["source"], distribution=[0.5])
        
        # Select 10 with 40% source, 30% title, 30% random others
        select_few_shot(examples, 10, method="distributed",
                       list_of_labels=["source", "title"], distribution=[0.4, 0.3])
    """
    if method == "order":
        if n >= len(examples):
            return examples
        else:
            return examples[:n]
    
    if method == "random":
        if n >= len(examples):
            return examples
        else:
            return random.sample(examples, n)
    
    if method == "distributed":
        if not list_of_labels or not distribution:
            raise ValueError("method='distributed' requires list_of_labels and distribution parameters")
        
        if len(distribution) != len(list_of_labels):
            raise ValueError(f"distribution length ({len(distribution)}) must match list_of_labels length ({len(list_of_labels)})")
        
        dist_sum = sum(distribution)
        if dist_sum > 1.0:
            raise ValueError(f"distribution sum cannot exceed 1.0, got {dist_sum}")
        
        # Categorize examples by labels
        categorized = {label: [] for label in list_of_labels}
        categorized["other"] = []
        
        for example in examples:
            _, output_text = example
            output_tokens = tokenize(output_text)
            labels_in_example = []
            for token in output_tokens:
                if is_auto_label_tag(token) == 1:
                    token_label = HTMLLabel(token)
                    labels_in_example.append(token_label.name)
                
                elif token.startswith('<') and token.endswith('>'):
                    # Could be a simplified tag
                    simple_label = from_simplified(token)
                    labels_in_example.append(simple_label.name)
            
            
            # Check if example contains any of the target labels
            found = False
            for target_label in list_of_labels:
                if target_label in labels_in_example:
                    categorized[target_label].append(example)
                    found = True
                    break  # Only categorize by first matching label
            
            if not found:
                categorized["other"].append(example)
        
        # Select examples according to distribution
        selected = []
        
        # First, select from specified labels
        for i, label in enumerate(list_of_labels):
            count = int(n * distribution[i])
            available = categorized[label]
            
            if count > len(available):
                print(f"   ⚠ Warning: Requested {count} examples with label '{label}', but only {len(available)} available")
                selected.extend(available)
            else:
                selected.extend(random.sample(available, count))
        
        # Fill remaining with "other" (automatically if distribution sum < 1.0)
        remaining = n - len(selected)
        if remaining > 0:
            available_other = categorized["other"]
            if remaining > len(available_other):
                selected.extend(available_other)
            else:
                selected.extend(random.sample(available_other, remaining))
        
        # Shuffle to mix the categories
        random.shuffle(selected)
        
        return selected[:n]  # Ensure we return exactly n examples
    
    raise ValueError(f"Unknown method: {method}")


In [253]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples_from_labels(fs_tokens, 
                                              sublabel_config)


# Select examples with distributed method: 50% with "source" label, 50% random others
selected_few_shot_examples = select_few_shot(
    examples=few_shot_examples, 
    n=n_few_shot,
    method="distributed",
    list_of_labels=sublabel_config["new_labels"],
    distribution=[0.8]
)
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")


   ✓ Selected 10 few-shot examples for processing.


In [254]:
selected_few_shot_examples

[('<decision>(1998), 37 O.R. (3d) 790</decision>',
  '<decision>(1998), 37 O.R. (3d) 790</decision>'),
 ('<decision>Hickman v. Taylor, 329 U.S. 495 (1946)</decision>',
  '<decision><title>Hickman v. Taylor</title>, 329 U.S. 495 (1946)</decision>'),
 ('<decision>Metropolitan Life Insurance Co. v. Frenette, [1992] 1 S.C.R. 647, 89 D.L.R. (4th) 653, 134 N.R. 169, [1992] I.L.R. 1-2823</decision>',
  '<decision><title>Metropolitan Life Insurance Co. v. Frenette</title>, [1992] 1 S.C.R. 647, 89 D.L.R. (4th) 653, 134 N.R. 169, [1992] I.L.R. 1-2823</decision>'),
 ('<secondary sources>Watson and Au, "Solicitor-Client Privilege and Litigation   Privilege in Civil Litigation" (1998), 77 Can. Bar Rev. 315,   pp. 333-35, 344-45, 346-49</secondary sources>',
  '<secondary sources>Watson and Au, "<title>Solicitor-Client Privilege and Litigation   Privilege in Civil Litigation</title>" (1998), 77 Can. Bar Rev. 315,   pp. 333-35, 344-45, 346-49</secondary sources>'),
 ('<decision>R. v. Seaboyer; R. v. 

In [258]:
prompt_path = fr"{project_root}\llm_based_annotation\utils\prompts\simplified_sublabels_extraction_from_parent_cot.txt"
pr = get_prompt_sublabel_extraction(prompt_path=prompt_path, keep_labels=sublabel_config["new_labels"], few_shot_examples=selected_few_shot_examples)



In [259]:
print(pr[0])

You are an advanced Language Model designed to annotate Canadian legal texts by identifying and marking mentions of legal authorities in judicial decisions.

You must follow the instructions below with extreme care.
Pay very close attention to the provided examples — they define the expected behavior.

---

## Task Objective

You are given a mention already wrapped in a parent label (e.g., <decision>...</decision>,  <legisltation>...</legislation>, <secondary sources>...</secondary sources>).

Your task is to identify and annotate **only the following sublabels** inside the mention:

title

You must NOT annotate anything else.

---

## Allowed Sublabel Definitions

- <title>: Official title or alias designating a legal authority, including a legislative text, a judicial decision, or a secondary source publication. For decisions: party names and the 'v.' formulation. For legislation: the name of the statute or regulation. For secondary sources: the title of the book, article, or specifi

In [260]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name)

In [256]:
SUBLABEL_DEFINITIONS = {
    "title": (
        "<title>: Official title or alias designating a legal authority, including "
        "a legislative text, a judicial decision, or a secondary source publication. "
        "For decisions: party names and the 'v.' formulation. "
        "For legislation: the name of the statute or regulation. "
        "For secondary sources: the title of the book, article, or specific contribution. "
        "Do NOT include publication venue or collection titles."
    ),

    "reference": (
        "<reference>: Bibliographic or publication information identifying a legislative "
        "text or a judicial decision, such as year, reporter, volume, or number "
        "(e.g., S.C.R., D.L.R., statute year and chapter). "
        "Applicable only to decisions and legislation."
    ),

    "source": (
        "<source>: Bibliographic or publication information identifying the source of a "
        "secondary publication, fully or partially. This may include journal name, "
        "collective work title, publisher, volume, year, or CanLIIdocs references. "
        "Applicable only to secondary sources. "
        "For collective works, include only the bibliographic information following "
        "'in' or 'dans', not the word itself."
    ),

    "authors": (
        "<authors>: Author or list of authors of a secondary source publication, "
        "including accompanying 'et al.' when present. "
        "All authors must be included within a single <authors> tag. "
        "Applicable only to secondary sources. "
        "Do NOT include scientific editors of collective works unless they are explicitly "
        "identified as authors of the cited contribution."
    ),

    "fragment": (
        "<fragment>: A precise part of a legal text, decision, or secondary source "
        "used to locate specific information, such as article, paragraph, page, "
        "section, or subsection number. Applicable to all authority types."
    ),
}



def build_sublabel_definitions(keep_labels):
    missing = [lbl for lbl in keep_labels if lbl not in SUBLABEL_DEFINITIONS]
    if missing:
        raise ValueError(f"Unknown sublabels: {missing}")

    return "\n".join(
        f"- {SUBLABEL_DEFINITIONS[label]}"
        for label in keep_labels
    )

def get_prompt_sublabel_extraction(prompt_path, keep_labels, few_shot_examples=None):
    """
    Generate dynamic prompt for sublabel extraction using a TXT template.

    Args:
        keep_labels (List[str]): Sublabels to extract (e.g. ["title", "reference"])

    Returns:
        Tuple[str, str]: (system_prompt, user_prompt_template)
    """

    with open(prompt_path, 'r', encoding='utf-8') as f:
        template = f.read()

    # --- Build dynamic fields ---
    sublabels_str = ", ".join(keep_labels)
    sublabels_definition = build_sublabel_definitions(keep_labels)

    # --- Fill template ---
    system_prompt = template.format(
        sublabels=sublabels_str,
        sublabels_definition=sublabels_definition,
    )

    # Add few-shot examples if provided
    if few_shot_examples:
        system_prompt += "\n\nHere are some examples:\n"
        for i, (input_text, expected_output) in enumerate(few_shot_examples, 1):
            system_prompt += f"\nExample {i}:\n"
            system_prompt += f"<ORIGINAL_TEXT>{input_text}<END_ORIGINAL_TEXT>\n"
            system_prompt += f"<EXPECTED_OUTPUT>{expected_output}<END_EXPECTED_OUTPUT>\n"
    
    user_prompt_template = """Please annotate the following legal text with the appropriate sublabels tags:

    <ORIGINAL_TEXT>{text}<END_ORIGINAL_TEXT>
    
    OUTPUT:"""

    return system_prompt, user_prompt_template

In [257]:
import os
import json
from tqdm import tqdm
from typing import List, Tuple, Optional


from utils import apply_post_processing_transforms
from utils import distance_lists_auto_label, apply_operations_safe
from utils import verify_processed_chunk


def _build_processing_segments(tokens, parent_mentions):
    """
    Build a list of segments alternating between:
      - non-processable token spans
      - processable mention spans

    Returns:
        List[dict]: each dict has:
            - "process": bool
            - "tokens": list
            - "meta": optional mention metadata
    """
    segments = []
    cursor = 0

    for html_label, start_idx, end_idx in parent_mentions:
        # Non-processable tokens before the mention
        if cursor < start_idx:
            segments.append({
                "process": False,
                "tokens": tokens[cursor:start_idx]
            })

        # The mention itself (processable)
        segments.append({
            "process": True,
            "tokens": tokens[start_idx:end_idx + 1],
            "meta": {
                "label": html_label,
                "start": start_idx,
                "end": end_idx
            }
        })

        cursor = end_idx + 1

    # Trailing non-processable tokens
    if cursor < len(tokens):
        segments.append({
            "process": False,
            "tokens": tokens[cursor:]
        })

    return segments


def process_single_mention(
    model,
    mention: list,
    system_prompt: str,
    user_prompt_template: str,
    sublabel_config: dict,
    allowed_labels: list = None,
    max_fallback_attempts: int = 1
):
    """
    Process a single parent mention to extract sublabels.
    
    Pipeline:
    1. Prepare input (simplified form, keep right attributes)
    2. Decode mention to text
    3. Generate LLM output
    4. Post-process output (TODO: extract, transform)
    5. Verify output (hallucination, consistency, label scheme)
    6. If verification fails, apply fallback
    7. If still fails, return original mention
    
    Args:
        model: LLM model instance
        mention: List of tokens for the parent mention
        system_prompt: System prompt for LLM
        user_prompt_template: User prompt template with {text} placeholder
        sublabel_config: Configuration for sublabel transformations
        allowed_labels: List of allowed sublabel names
        max_fallback_attempts: Maximum number of fallback attempts
    
    Returns:
        Tuple of (processed_tokens, status, error_details)
    """
    # ------ 1. PREPARE INPUT ------
    input_config = sublabel_config.copy()
    input_config["keep_labels"] = None # Keep everything
    input_config["switch_type"] = False # Keep original types for input
    prepared_mention = prepare_label_tokens(mention,
        label_config={
            "switch_type": False,
            "use_simplified": sublabel_config.get("use_simplified", False),
            "keep_attributes": sublabel_config.get("keep_attributes", None)
        }
    )

    
    # ------ 2. DECODE TO TEXT ------
    text = decode(prepared_mention)
    
    # ------ 3. GENERATE LLM OUTPUT ------
    user_prompt = user_prompt_template.format(text=text)
    raw_output = model.generate(
        system_prompt=system_prompt,
        user_prompt=user_prompt
    )
    
    # ------ 4. POST-PROCESS OUTPUT ------
    try:
        processed_tokens = apply_post_processing_transforms(
            raw_output=raw_output,
            use_simplified=sublabel_config.get("use_simplified", False), # Did we use simplified form ?
            label_type='auto_label'
        )
    except Exception as e:
        return mention, "Post-processing Error", f"Failed to post-process: {str(e)}"
    

    
    # ------ 5. APPLY ERROR CORRECTION ------
    # This aligns tokens to handle minor discrepancies
    cleaned_input_mention_token = prepare_label_tokens(
        mention,
        label_config={
            "keep_attributes": sublabel_config.get("keep_attributes", None),
            "switch_type": False,
            "use_simplified": False
        }
    ) # Just remove non-kept attributes for alignment
    _, operations = distance_lists_auto_label(cleaned_input_mention_token, processed_tokens)
    processed_tokens_corrected = apply_operations_safe(processed_tokens, operations)

    #print(f"========== DEBUG DISTANCE INPUT : {cleaned_input_mention_token}")
    #print(f"========== DEBUG DISTANCE OUTPUT : {processed_tokens_corrected}")
    


    
    # ------ 6. VERIFY OUTPUT ------
    verification = verify_processed_chunk(
        original_tokens=mention,
        processed_tokens=processed_tokens_corrected,
        allowed_labels=allowed_labels,
        check_scheme=True
    )
    
    if verification.passed:
        return processed_tokens_corrected, "Success", None
    
    else :
        return mention, "Verification Failed", verification.details


def process_labels(
    model,
    tokens: list,
    sublabel_config: dict,
    few_shot_examples: list = None,
    prompt_path: str = None,
    output_dir: str = None,
    filename: str = None,
    max_fallback_attempts: int = 1
):
    """
    Process tokens to extract sublabels from parent mentions.
    
    This function extracts sublabels (e.g., title) from already annotated
    parent labels (e.g., decision, legislation, secondary sources).
    
    Args:
        model: LLM model instance
        tokens: List of tokens containing parent labels
        sublabel_config: Configuration dict with:
            - parent: List of parent label names
            - keep_labels: List of sublabel names to extract
            - keep_attributes, switch_type, use_simplified: Transform options
        few_shot_examples: Optional list of (input, output) examples
        prompt_path: Optional path to prompt templates
        output_dir: Optional directory to save outputs
        filename: Optional filename prefix for outputs
        max_fallback_attempts: Maximum fallback attempts per error type
    
    Returns:
        List of processed tokens (flat list, not chunked)
    """
    # ------ 1. GET PROMPT ------
    system_prompt, user_prompt_template = get_prompt_sublabel_extraction(
        prompt_path=prompt_path,
        keep_labels=sublabel_config["new_labels"],
        few_shot_examples=few_shot_examples
    )
    
    # ------ 2. GET LIST OF AUTO_LABEL MENTIONS TO PROCESS ------
    # We're looking for auto_label parents (already extracted from previous step)
    parent_mentions = get_list_of_mention(
        tokens=tokens,
        keep_labels=sublabel_config["parent"],
        label_type="auto_label"  # Process auto_labels from parent extraction
    )
    
    print(f"   ✓ Found {len(parent_mentions)} parent mentions to process")
    
    # ------ 3. INITIALIZE TRACKING ------
    from process_chunks import ProcessingHistory
    history = ProcessingHistory()
    
    # Create a copy of tokens to modify
    processed_tokens = tokens.copy()
    
    # ------ 4. BUILD SEGMENTS ------
    segments = _build_processing_segments(tokens, parent_mentions)

    print(f"   ✓ Built {len(segments)} token segments "
        f"({sum(s['process'] for s in segments)} to process)")

    # ------ 5. PROCESS EACH PROCESSABLE SEGMENT ------
    for idx, segment in enumerate(tqdm(segments, desc="Processing mentions")):
        if not segment["process"]:
            continue

        mention = segment["tokens"]
        html_label = segment["meta"]["label"]

        processed_mention, status, error_details = process_single_mention(
            model=model,
            mention=mention,
            system_prompt=system_prompt,
            user_prompt_template=user_prompt_template,
            sublabel_config=sublabel_config,
            allowed_labels=sublabel_config["new_labels"] + sublabel_config["already_labeled"] + sublabel_config["parent"],
            max_fallback_attempts=max_fallback_attempts
        )

        # Replace the entire segment safely
        segment["tokens"] = processed_mention

        # Track history
        history.add(
            status,
            idx,
            decode(processed_mention),
            error_details
        )

        if status != "Success" and not status.startswith("Success (after"):
            print(f"   ⚠ Segment {idx} ({html_label.name}) failed: {status}")

    
    # ------ 6. SAVE HISTORY AND RESULTS ------
    if output_dir and filename:
        history.save(output_dir, f"{filename}_sublabel")
        
        # Save processed tokens
        json_path = os.path.join(output_dir, f"processed_sublabels_{filename}.json")
        try:
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(decode(processed_tokens), f, indent=4, ensure_ascii=False)
            print(f"   ✓ Processed tokens saved to: {json_path}")
        except Exception as e:
            print(f"   ✗ Error saving processed tokens: {e}")
    
    # ------ 7. PRINT SUMMARY ------
    summary = history.summary()
    print(f"\n   ✓ Sublabel extraction completed:")
    print(f"      - Total mentions: {summary['total']}")
    print(f"      - Successful: {summary['success']}")
    print(f"      - Failed: {summary['total'] - summary['success']}")



    # ------ 8. FLATTEN SEGMENTS ------
    processed_tokens = [
        token
        for segment in segments
        for token in segment["tokens"]
    ]
    
    return processed_tokens

In [ ]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils\prompts\simplified_parent_extraction_cot.txt"

processed_label = process_labels(
    model=model,
    tokens=tokens,
    sublabel_config=sublabel_config,
    few_shot_examples=few_shot_examples,
    prompt_path=prompt_path,
    output_dir=output_dir,
    filename=filename,
    max_fallback_attempts=1
)


   ✓ Found 598 parent mentions to process
   ✓ Built 1196 token segments (598 to process)


Processing mentions:   0%|          | 2/1196 [00:03<33:02,  1.66s/it]

   → Step 1: Tokenized into 23 tokens
   ✓ Extracted 21 tokens between <start> and <end>
   → Step 2: Extracted 21 tokens between markers
   ✓ Converted 21 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


Processing mentions:   0%|          | 4/1196 [00:05<23:44,  1.20s/it]

   → Step 1: Tokenized into 24 tokens
   ✓ Extracted 22 tokens between <start> and <end>
   → Step 2: Extracted 22 tokens between markers
   ✓ Converted 22 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


...

   → Step 1: Tokenized into 11 tokens
   ✓ Extracted 9 tokens between <start> and <end>
   → Step 2: Extracted 9 tokens between markers
   ✓ Converted 9 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


Processing mentions:  76%|███████▌  | 907/1196 [14:45<04:03,  1.18it/s]

   → Step 1: Tokenized into 11 tokens
   ✓ Extracted 9 tokens between <start> and <end>
   → Step 2: Extracted 9 tokens between markers
   ✓ Converted 9 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


Processing mentions:  76%|███████▌  | 909/1196 [14:47<03:57,  1.21it/s]

   → Step 1: Tokenized into 19 tokens
   ✓ Extracted 17 tokens between <start> and <end>
   → Step 2: Extracted 17 tokens between markers
   ✓ Converted 17 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


In [217]:
test1 = []
test2 = []
for token in processed_label:
    if not is_auto_label_tag(token) in [1, 2]:
        test1.append(token)


for token in tokens:
    if not is_auto_label_tag(token) in [1, 2]:
        test2.append(token)


In [218]:
print(test1)
print(test2)

['<!DOCTYPE html>', '\n\n', '<html>', '<!-- HTMLLabelizer\n{\n  "labeltree": {\n    "legislation": {\n      "color": "#76CEDE",\n      "sublabels": {\n        "title": {\n          "color": "#93c47d",\n          "sublabels": {},\n          "attributes": {\n            "titletype": {\n              "type": "dropdown",\n              "options": [\n                "official",\n                "alias"\n              ],\n              "default": "official",\n              "groupRole": "regular"\n            }\n          }\n        },\n        "reference": {\n          "color": "#ff5733",\n          "sublabels": {},\n          "attributes": {}\n        },\n        "fragment": {\n          "color": "#8e7cc3",\n          "sublabels": {},\n          "attributes": {\n            "fragmentid": {\n              "type": "string",\n              "default": "",\n              "groupRole": "regular"\n            },\n            "non_standard": {\n              "type": "checkbox",\n              "defau

In [219]:
print(test1 == test2)

True


## Post Processing

In [220]:

processed_html = decode(processed_label)

print(f"\nMerged HTML length: {len(processed_html)}")

# ---------- Add style and parent to auto_label tags ----------
processed_html_content = decode(add_style_and_parent_to_auto_labels(processed_html))







Merged HTML length: 262475


In [221]:
# ---------- Compare with original HTML (ignoring auto_label tags) ----------
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)

   ✓ HTMLs match when ignoring auto_label tags


In [222]:
# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_llm_{anno}_{out_version}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\1999CanLII7320_annotated_llm_EG
